In [ ]:
import pandas as pd
import regex as re
from pathlib import Path
from collections import Counter, defaultdict
import itertools
from tqdm import tqdm

# NLP tools
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import spacy
nlp = spacy.load("en_core_web_sm")

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

PS = PorterStemmer()
WL = WordNetLemmatizer()

In [ ]:
INPUT_CSV = "Shakespeare.csv"
HAS_LINES = True          # True if dataset has full lines (PlayerLine). False if only tokens.
TEXT_COL = "PlayerLine"   # column with original text/lines
TOKEN_COL = "token"       # column name if tokens-only
OUT_DIR = Path("module2_output")
OUT_DIR.mkdir(exist_ok=True)

# Utility: normalize token string
def normalize(s):
    if pd.isna(s): return ""
    return re.sub(r'\s+', ' ', str(s).strip()).lower()

In [ ]:
df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
if HAS_LINES:
    df['text_norm'] = df[TEXT_COL].apply(normalize)
    # drop empty
    df = df[df['text_norm'].str.len() > 0].copy()
    corpus_lines = df['text_norm'].tolist()
else:
    # tokens only
    df['token_norm'] = df[TOKEN_COL].apply(normalize)
    df = df[df['token_norm'] != ""].copy()
    # tokens as list
    tokens = df['token_norm'].tolist()
    corpus_lines = None

In [ ]:
morph_rows = []
if HAS_LINES:
    # work at token level but preserve line context
    for line in corpus_lines:
        toks = nltk.word_tokenize(line)
        for tok in toks:
            stem = PS.stem(tok)
            # lemmatize via WordNet if possible; spaCy alternative below
            lemma = WL.lemmatize(tok)
            morph_rows.append({'token': tok, 'stem': stem, 'lemma_wordnet': lemma})
else:
    for tok in tokens:
        stem = PS.stem(tok)
        lemma = WL.lemmatize(tok)
        morph_rows.append({'token': tok, 'stem': stem, 'lemma_wordnet': lemma})

In [ ]:
morph_df = pd.DataFrame(morph_rows)
# optionally add spaCy lemma and POS
morph_df['lemma_spacy'] = [nlp(t)[0].lemma_ if t else '' for t in morph_df['token']]
morph_df['pos_spacy'] = [nlp(t)[0].pos_ if t else '' for t in morph_df['token']]

morph_df.to_csv(OUT_DIR / "morphology_table.csv", index=False, encoding='utf-8')

In [ ]:
def generate_ngrams_from_lines(lines, n, word_level=True):
    counts = Counter()
    if word_level:
        for line in lines:
            toks = nltk.word_tokenize(line)
            if len(toks) < n: continue
            for i in range(len(toks)-n+1):
                ngram = tuple(toks[i:i+n])
                counts[ngram] += 1
    else:
        # char-level: join lines with space preserved or removed depending on need
        for line in lines:
            seq = line.replace(" ", "")  # or keep spaces if you want ' ' as char
            if len(seq) < n: continue
            for i in range(len(seq)-n+1):
                ngram = tuple(seq[i:i+n])
                counts[ngram] += 1
    return counts

In [ ]:
def generate_ngrams_from_tokens(tokens, n, word_level=True, window_size=None):
    # tokens: list of isolated tokens (no sentence separation)
    counts = Counter()
    if window_size is None:
        # treat tokens as one long stream
        seq = tokens
        if len(seq) >= n:
            for i in range(len(seq)-n+1):
                ngram = tuple(seq[i:i+n])
                counts[ngram] += 1
    else:
        # sliding windows: create pseudo-sentences of length=window_size
        for i in range(0, max(1, len(tokens)-window_size+1), window_size):
            block = tokens[i:i+window_size]
            if len(block) < n:
                continue
            for j in range(len(block)-n+1):
                ngram = tuple(block[j:j+n])
                counts[ngram] += 1
    return counts

In [ ]:
n_values = [1,2,3]
for n in n_values:
    if HAS_LINES:
        wc = generate_ngrams_from_lines(corpus_lines, n, word_level=True)
        cc = generate_ngrams_from_lines(corpus_lines, n, word_level=False)
    else:
        # tokens-only: use window_size to create pseudo-sentences (choose 5-10)
        wc = generate_ngrams_from_tokens(tokens, n, word_level=True, window_size=8)
        cc = generate_ngrams_from_tokens(tokens, n, word_level=False, window_size=50)  # char windows bigger

    # save word-level ngrams
    wrows = [{'ngram': ' '.join(k), 'count': v} for k,v in wc.items()]
    pd.DataFrame(wrows).sort_values('count', ascending=False).to_csv(
        OUT_DIR / f"ngram_counts_word_{n}.csv", index=False, encoding='utf-8'
    )
    crows = [{'ngram': ''.join(k), 'count': v} for k,v in cc.items()]
    pd.DataFrame(crows).sort_values('count', ascending=False).to_csv(
        OUT_DIR / f"ngram_counts_char_{n}.csv", index=False, encoding='utf-8'
    )

In [ ]:
bigram_counts = Counter()
unigram_counts = Counter()
if HAS_LINES:
    for line in corpus_lines:
        toks = nltk.word_tokenize(line)
        for i in range(len(toks)):
            unigram_counts[(toks[i],)] += 1
            if i < len(toks)-1:
                bigram_counts[(toks[i], toks[i+1])] += 1
else:
    # tokens-only stream approach
    for i in range(len(tokens)):
        unigram_counts[(tokens[i],)] += 1
        if i < len(tokens)-1:
            bigram_counts[(tokens[i], tokens[i+1])] += 1

In [ ]:
prob_rows = []
for (w1,w2), cnt in bigram_counts.items():
    denom = unigram_counts[(w1,)]
    prob = cnt / denom if denom > 0 else 0.0
    prob_rows.append({'w1': w1, 'w2': w2, 'count': cnt, 'count_w1': denom, 'P(w2|w1)': prob})

pd.DataFrame(prob_rows).sort_values('P(w2|w1)', ascending=False).to_csv(
    OUT_DIR / "bigram_probabilities.csv", index=False, encoding='utf-8'
)

In [ ]:
def predict_next_word(w1, topk=5):
    rows = [r for r in prob_rows if r['w1']==w1]
    rows_sorted = sorted(rows, key=lambda x: x['P(w2|w1)'], reverse=True)
    return rows_sorted[:topk]

In [ ]:
import json
pred_dump = defaultdict(list)
for r in prob_rows:
    pred_dump[r['w1']].append((r['w2'], r['P(w2|r1)']) if False else (r['w2'], r['P(w2|w1)']))
# Save top 5 per w1
pred_top = {k: sorted(v, key=lambda x: x[1], reverse=True)[:5] for k,v in pred_dump.items()}
with open(OUT_DIR / "predictor_bigram_top5.json", "w", encoding='utf-8') as f:
    json.dump(pred_top, f, ensure_ascii=False, indent=2)

In [ ]:
pos_rows = []
if HAS_LINES:
    for line in tqdm(corpus_lines, desc="POS tagging lines"):
        doc = nlp(line)
        for token in doc:
            pos_rows.append({'token': token.text, 'lemma': token.lemma_, 'pos': token.pos_, 'tag': token.tag_, 'line': line})
else:
    # tokens-only: tag each token in isolation and optionally with a window
    # simple isolated tagging
    for tok in tokens:
        doc = nlp(tok)
        t = doc[0]
        pos_rows.append({'token': tok, 'lemma': t.lemma_, 'pos': t.pos_, 'tag': t.tag_, 'line': ''})

In [ ]:
pos_df = pd.DataFrame(pos_rows)
pos_df.to_csv(OUT_DIR / "pos_tagged.csv", index=False, encoding='utf-8')

print("Module 2 outputs written to:", OUT_DIR)